# 01 · Data Profiling, Cleaning & Star Schema

**Goal:** turn 1.07M raw transaction lines from the UCI *Online Retail II* dataset
into a clean, analysis-ready **star schema** (facts + dimensions) that feeds
PostgreSQL, the RFM/churn models and the Power BI report.

Every cleaning decision below is *evidence first*: show the problem, quantify it,
then fix it. All decisions are logged to `reports/data_quality_log.csv`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src import cleaning as cl
from src import config as C

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

## 1. Load both yearly sheets

In [2]:
raw = cl.load_raw()
print(f"{len(raw):,} rows × {raw.shape[1]} columns")
print("Date range:", raw["invoice_datetime"].min(), "→", raw["invoice_datetime"].max())
raw.head()

1,067,371 rows × 9 columns
Date range: 2009-12-01 07:45:00 → 2011-12-09 12:50:00


,invoice_no,stock_code,description,quantity,invoice_datetime,unit_price,customer_id,country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


## 2. Profiling: what is wrong with this data?

In [3]:
profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_%": (raw.isna().mean() * 100).round(2),
    "unique": raw.nunique(),
})
profile

,dtype,missing,missing_%,unique
invoice_no,str,0,0.00,53628
stock_code,str,0,0.00,5304
description,string,4382,0.41,5631
quantity,int64,0,0.00,1057
invoice_datetime,datetime64[ns],0,0.00,47635
unit_price,float64,0,0.00,2807
customer_id,float64,243007,22.77,5942
country,str,0,0.00,43
source_sheet,str,0,0.00,2


### Issue 1: the two Excel sheets overlap
Sheet *Year 2009-2010* runs until **2010-12-09**, but sheet *Year 2010-2011*
starts on **2010-12-01**. Nine days of transactions are present twice.
Many public analyses of this dataset silently double-count this period.

In [4]:
for sheet, part in raw.groupby("source_sheet"):
    print(f"{sheet}: {part['invoice_datetime'].min()} → {part['invoice_datetime'].max()}")

s1 = raw[(raw["source_sheet"] == C.SHEETS[0]) & (raw["invoice_datetime"] >= C.SHEET_OVERLAP_START)]
s2 = raw[(raw["source_sheet"] == C.SHEETS[1]) & (raw["invoice_datetime"] < "2010-12-10")]
print(f"\nOverlap rows: sheet1={len(s1):,}, sheet2={len(s2):,}")
print("Identical invoice sets:", set(s1["invoice_no"]) == set(s2["invoice_no"]))
print(f"Revenue that would be double-counted: {C.CURRENCY}{(s1.quantity * s1.unit_price).sum():,.0f}")

Year 2009-2010: 2009-12-01 07:45:00 → 2010-12-09 20:01:00
Year 2010-2011: 2010-12-01 08:26:00 → 2011-12-09 12:50:00

Overlap rows: sheet1=22,523, sheet2=22,523
Identical invoice sets: True
Revenue that would be double-counted: £377,488


### Issue 2: cancellations, negative quantities and zero prices

In [5]:
is_cancel = raw["invoice_no"].str.startswith("C")
summary = pd.Series({
    "Cancelled lines (invoice starts with 'C')": is_cancel.sum(),
    "Non-cancelled lines with quantity <= 0": ((raw.quantity <= 0) & ~is_cancel).sum(),
    "Lines with price = 0": (raw.unit_price == 0).sum(),
    "Lines with price < 0 ('A' bad-debt adjustments)": (raw.unit_price < 0).sum(),
    "Exact duplicate rows (all columns)": raw.drop(columns="source_sheet").duplicated().sum(),
})
summary.to_frame("rows")

,rows
Cancelled lines (invoice starts with 'C'),19494
Non-cancelled lines with quantity <= 0,3457
Lines with price = 0,6202
Lines with price < 0 ('A' bad-debt adjustments),5
Exact duplicate rows (all columns),34335


### Issue 3: stock codes that are not products
Postage (`POST`, `DOT`), manual entries (`M`), bank charges, Amazon fees,
gift vouchers and test codes would distort product rankings.

In [6]:
nonprod = raw[cl.is_non_product(raw["stock_code"])]
(nonprod.assign(value=nonprod.quantity * nonprod.unit_price)
        .groupby("stock_code")
        .agg(lines=("invoice_no", "size"), value=("value", "sum"),
             example=("description", "first"))
        .sort_values("lines", ascending=False)
        .head(12))

,lines,value,example
stock_code,,,
POST,2122,112341.000,POSTAGE
DOT,1446,322647.470,DOTCOM POSTAGE
M,1421,-82796.320,MANUAL
C2,282,13386.000,CARRIAGE
D,177,-13484.540,DISCOUNT
S,104,-6065.800,SAMPLES
BANK CHARGES,102,-35562.629,BANK CHARGES
ADJUST,67,6835.240,ADJUSTMENT BY JOHN ON 26/01/2010 16
AMAZONFEE,43,-260763.580,AMAZON FEE


### Issue 4: missing Customer IDs
~23% of lines have no Customer ID (guest checkouts). Dropping them, as most
tutorials do, would understate revenue. **Decision:** keep them under a guest
key (`0`) for sales KPIs, exclude them from customer analytics (RFM, churn, cohorts).

In [7]:
missing_rev = raw.loc[raw["customer_id"].isna()]
print(f"Lines without Customer ID: {len(missing_rev):,} ({len(missing_rev)/len(raw):.1%})")

Lines without Customer ID: 243,007 (22.8%)


### Issue 5: inconsistent product descriptions & country names

In [8]:
desc_variants = raw.groupby("stock_code")["description"].nunique()
print(f"Stock codes with more than one description: {(desc_variants > 1).sum():,}")
example_code = desc_variants.sort_values(ascending=False).index[0]
print(f"\nExample — {example_code}:")
print(raw.loc[raw.stock_code == example_code, "description"].value_counts().head(6))
print("\nCountry labels needing fixes:", sorted(set(raw["country"]) & set(C.COUNTRY_FIXES)))

Stock codes with more than one description: 1,212

Example — 20713:
description
JUMBO BAG OWLS                  1372
FOUND                              2
MISSING                            1
WRONGLY MARKED. 23343 IN BOX       1
WRONGLY CODED-23343                1
WRONGLY MARKED 23343               1
Name: count, dtype: int64[pyarrow]



Country labels needing fixes: ['EIRE', 'European Community', 'Korea', 'RSA', 'USA', 'West Indies']


## 3. Apply the cleaning pipeline
Logic lives in `src/cleaning.py` so it is testable and reusable.

In [9]:
sales, returns, fees, quality_log = cl.clean(raw)
quality_log

,step,rows_affected,rows_remaining,line_value_remaining,note
0,Raw data loaded (both sheets),0,1067371,19287250.57,
1,Removed overlap between the two Excel sheets (...,22523,1044848,18909762.12,Same invoices appear in both yearly sheets
2,Removed exact duplicate rows,11812,1033036,18855533.70,
3,"Separated non-product lines (postage, fees, ad...",5818,1027218,18927435.65,Saved to fact_fees; excluded from product anal...
4,Separated cancelled/returned lines (invoice st...,17914,1009304,19643861.62,Saved to fact_returns
5,Removed lines with quantity <= 0 or price <= 0,5964,1003340,19643861.62,Stock adjustments / write-offs / zero-price sa...
6,Mapped missing Customer IDs to guest key 0 (ke...,226761,1003340,19643861.62,"Excluded from customer-level analysis (RFM, ch..."
7,"Standardised country names (EIRE->Ireland, RSA...",0,1003340,19643861.62,


### Extreme orders that were immediately cancelled
The two largest sales lines (80,995 and 74,215 units) were reversed within
minutes. Because the returns table keeps these reversals, **net revenue**
(sales − returns) neutralises them. We report net revenue as the headline KPI.

In [10]:
top = sales.nlargest(3, "line_value")[["invoice_no", "invoice_datetime", "customer_id", "stock_code", "quantity", "line_value"]]
top_ret = returns.nlargest(3, "line_value")[["invoice_no", "invoice_datetime", "customer_id", "stock_code", "quantity", "line_value"]]
display(top)
display(top_ret)

,invoice_no,invoice_datetime,customer_id,stock_code,quantity,line_value
1065882,581483,2011-12-09 09:15:00,16446,23843,80995,168469.6
587080,541431,2011-01-18 10:01:00,12346,23166,74215,77183.6
748132,556444,2011-06-10 15:28:00,15098,22502,60,38970.0


,invoice_no,invoice_datetime,customer_id,stock_code,quantity,line_value
1065883,C581484,2011-12-09 09:27:00,16446,23843,80995,168469.6
587085,C541433,2011-01-18 10:17:00,12346,23166,74215,77183.6
685606,C550456,2011-04-18 13:08:00,15749,21108,3114,6539.4


## 4. Build the star schema

| Table | Grain | Purpose |
|---|---|---|
| `fact_sales` | one invoice line | revenue, quantity, cohort index |
| `fact_returns` | one cancelled line | returns value & rate |
| `fact_fees` | one non-product line | postage/fees (kept for completeness) |
| `dim_customer` | one customer | home country, first/last purchase, cohort |
| `dim_product` | one stock code | canonical description, derived category |
| `dim_date` | one day | calendar attributes |
| `dim_country` | one country | ISO code, region, coordinates for the map |

In [11]:
dim_product = cl.build_dim_product(sales, returns)
dim_customer = cl.build_dim_customer(sales, returns)
dim_date = cl.build_dim_date(raw["invoice_datetime"].min(), raw["invoice_datetime"].max())
dim_country = cl.build_dim_country(sales, returns, fees, dim_customer.rename(columns={"home_country": "country"}))

fact_sales = cl.finalize_fact(sales, "sales_line_id", dim_customer)
fact_returns = cl.finalize_fact(returns, "return_line_id")
fact_fees = cl.finalize_fact(fees.assign(line_value=(fees.quantity * fees.unit_price).round(2)), "fee_line_id")

for name, t in {"fact_sales": fact_sales, "fact_returns": fact_returns, "fact_fees": fact_fees,
                "dim_customer": dim_customer, "dim_product": dim_product,
                "dim_date": dim_date, "dim_country": dim_country}.items():
    print(f"{name:<13} {len(t):>9,} rows")

fact_sales    1,003,340 rows
fact_returns     17,914 rows
fact_fees         5,818 rows
dim_customer      5,876 rows
dim_product       4,908 rows
dim_date            739 rows
dim_country          41 rows


### Product categories (derived)
The dataset has no category column, so categories are derived from description
keywords with ordered rules (`src/cleaning.py → CATEGORY_RULES`).
Only ~6% of revenue falls into *Other*.

In [12]:
cat_share = (fact_sales.merge(dim_product[["stock_code", "category"]], on="stock_code")
             .groupby("category")["line_value"].sum()
             .sort_values(ascending=False))
(cat_share / cat_share.sum() * 100).round(1).to_frame("revenue_share_%")

,revenue_share_%
category,
Kitchen & Dining,24.1
Home Decor,21.4
Bags & Storage,12.3
Lighting & Candles,9.0
Personal & Accessories,6.6
Party & Gift Wrap,5.9
Other,5.9
Christmas & Seasonal,5.2
Stationery & Crafts,4.3


## 5. Integrity checks before saving

In [13]:
assert fact_sales["sales_line_id"].is_unique
assert (fact_sales["quantity"] > 0).all() and (fact_sales["unit_price"] > 0).all()
assert set(fact_sales["stock_code"]) <= set(dim_product["stock_code"])
assert set(fact_sales["customer_id"]) <= set(dim_customer["customer_id"])
assert set(fact_returns["customer_id"]) <= set(dim_customer["customer_id"])
print(f"Return-only customers (purchase before Dec-2009): {(~dim_customer.has_purchases).sum()}")
assert set(fact_sales["country"]) <= set(dim_country["country"])
assert set(fact_sales["date_key"]) <= set(dim_date["date_key"])
assert fact_sales.loc[fact_sales.customer_id != 0, "cohort_index"].min() == 0
print("All referential-integrity checks passed ✔")

gross = fact_sales["line_value"].sum()
ret = fact_returns["line_value"].sum()
print(f"Gross sales {C.CURRENCY}{gross:,.0f} | Returns {C.CURRENCY}{ret:,.0f} "
      f"| Net {C.CURRENCY}{gross - ret:,.0f} | Return rate {ret / gross:.2%}")

Return-only customers (purchase before Dec-2009): 23


All referential-integrity checks passed ✔
Gross sales £19,643,862 | Returns £716,426 | Net £18,927,436 | Return rate 3.65%


## 6. Save

In [14]:
C.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
C.REPORTS_DIR.mkdir(parents=True, exist_ok=True)

date_fmt = "%Y-%m-%d %H:%M:%S"
# fact_sales is ~80 MB, so it is split by year to stay under GitHub's 50 MB
# soft limit. Power BI's "Folder" connector and src.io.load_fact_sales() recombine it.
sales_dir = C.PROCESSED_DIR / "fact_sales"
sales_dir.mkdir(exist_ok=True)
for year, part in fact_sales.groupby(fact_sales["invoice_datetime"].dt.year):
    part.to_csv(sales_dir / f"fact_sales_{year}.csv", index=False, date_format=date_fmt)
fact_returns.to_csv(C.PROCESSED_DIR / "fact_returns.csv", index=False, date_format=date_fmt)
fact_fees.to_csv(C.PROCESSED_DIR / "fact_fees.csv", index=False, date_format=date_fmt)
dim_customer.to_csv(C.PROCESSED_DIR / "dim_customer.csv", index=False, date_format="%Y-%m-%d")
dim_product.to_csv(C.PROCESSED_DIR / "dim_product.csv", index=False)
dim_date.to_csv(C.PROCESSED_DIR / "dim_date.csv", index=False, date_format="%Y-%m-%d")
dim_country.to_csv(C.PROCESSED_DIR / "dim_country.csv", index=False)
quality_log.to_csv(C.REPORTS_DIR / "data_quality_log.csv", index=False)
print("Saved to", C.PROCESSED_DIR)

Saved to /home/claude/retail-customer-analytics/data/processed
